In [1]:
!unzip -q sample50.zip -d /content/
!ls /content/sample50/images | head -3
!ls /content/sample50/masks | head -3
!ls /content/sample50/images | wc -l

10675.jpg
10684.jpg
11001.jpg
10675.webp
10684.webp
11001.webp
50


In [4]:
%pip install -q mediapipe

import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite",
    "/content/hair_segmenter.tflite",
)
print("Download Completed")

Download Completed


In [5]:
import glob, os
import numpy as np
from PIL import Image
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

options = vision.ImageSegmenterOptions(
    base_options=python.BaseOptions(model_asset_path="/content/hair_segmenter.tflite"),
    output_confidence_masks=True,
)
segmenter = vision.ImageSegmenter.create_from_options(options)

THRESHOLDS = [0.5, 0.6, 0.7, 0.75, 0.8, 0.9]
stats = {t: {"tp": 0, "fp": 0, "fn": 0} for t in THRESHOLDS}

img_paths = sorted(glob.glob("/content/sample50/images/*.jpg"))
print("No.Of images to evaluate:", len(img_paths))

for img_path in img_paths:
    name = os.path.splitext(os.path.basename(img_path))[0]
    mask_path = f"/content/sample50/masks/{name}.webp"

    img = Image.open(img_path).convert("RGB")

    # The white (bright value) is the hair. Resize to the image size to match the model input size.
    gt = np.array(Image.open(mask_path).convert("L").resize(img.size, Image.NEAREST)) > 127

    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img))
    prob = np.squeeze(segmenter.segment(mp_img).confidence_masks[1].numpy_view())

    # If the prediction probability mask size differs from the ground truth, it will be adjusted.
    if prob.shape != gt.shape:
        prob = np.array(Image.fromarray(prob).resize(img.size, Image.BILINEAR))

    for t in THRESHOLDS:
        pred = prob >= t
        stats[t]["tp"] += int((pred & gt).sum())
        stats[t]["fp"] += int((pred & ~gt).sum())
        stats[t]["fn"] += int((~pred & gt).sum())

print()
print("th     Precision  Recall   F1      IoU")
for t, s in stats.items():
    p = s["tp"] / (s["tp"] + s["fp"])
    r = s["tp"] / (s["tp"] + s["fn"])
    f1 = 2 * p * r / (p + r)
    iou = s["tp"] / (s["tp"] + s["fp"] + s["fn"])
    print(f"{t:<6} {p:.3f}      {r:.3f}    {f1:.3f}   {iou:.3f}")

No.Of images to evaluate: 50

th     Precision  Recall   F1      IoU
0.5    0.938      0.908    0.923   0.857
0.6    0.950      0.893    0.921   0.853
0.7    0.960      0.874    0.915   0.843
0.75   0.965      0.861    0.910   0.835
0.8    0.969      0.846    0.903   0.824
0.9    0.979      0.801    0.881   0.787
